# E1.9 · Model and agent lifecycle governance

**Function E — AI Governance for Agentic Systems → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.8 · Third-party and model supply chain risk](https://spbreed.github.io/cyber-commons/lessons/E1.8.html)**.

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Models and agents get approved once and then change forever. Without an explicit lifecycle — approval, change, revalidation, decommission — what you approved and what is running have no necessary relationship.

> **At CyberTravels.** TripBot was approved once and has changed continuously since — a tool added, a prompt edited, a model upgraded silently by the provider. None of it raised a ticket.

## 2 · The framework

```
   approved once                   changes forever
   +---------------+               prompt . model . tools . scope
   | v1, march     |  ---------->  ??? , ??? , ??? , ???
   +---------------+

   lifecycle: approve -> change control -> revalidate -> decommission
   without it, "what we approved" and "what is running" are unrelated
```

Lifecycle governance is about the events that have no ticket.

A model or agent has a lifecycle — requested, approved, deployed, changed,
retired. Classical governance covers the first, second and third. The events that
actually change your risk are the fourth and fifth, and they mostly happen
outside any process:

| Event | Ticketed? | Why it matters |
|---|---|---|
| new agent deployed | usually | caught by existing process |
| tool added to manifest | no | changes blast radius silently |
| prompt edited | no | changes behaviour, not code |
| provider upgrades the model | no | you may not be told |
| scope widened in IAM | sometimes | depends on your IAM review |
| **agent decommissioned** | rarely | **the identity outlives the agent** |

The last row is the one most first reviews find: a retired agent whose identity
still exists is a standing credential with no owner and nobody watching it,
because everyone believes it is gone.

## 3 · Demo — the lifecycle, and which events generate a record

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">lifecycle event</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">does it raise a ticket?</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">why it matters</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">new agent deployed</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">usually</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the existing change process catches it</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">tool added to the manifest</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">changes blast radius, and no pull request is raised</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">prompt edited in a console</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">changes behaviour, not code</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">provider upgrades the model</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">you may not be told at all</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">scope widened in IAM</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">sometimes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">depends entirely on your access-review cadence</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agent decommissioned</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>rarely</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the identity usually outlives the agent</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Four of six generate no reliable record. A lifecycle you cannot observe is a lifecycle you are not governing.</div>

## 4 · Where it breaks — the identity that outlived the agent

In [ ]:
import time
now = time.time(); DAY = 86400

IDENTITIES = {
 "triage-agent":   {"created": now - 200*DAY, "last_auth": now - 0.2*DAY,
                    "owner": "appsec", "service_running": True},
 "patch-agent":    {"created": now - 180*DAY, "last_auth": now - 1*DAY,
                    "owner": "platform", "service_running": True},
 "legacy-scanner": {"created": now - 900*DAY, "last_auth": now - 400*DAY,
                    "owner": "", "service_running": False},
 "poc-agent-2025": {"created": now - 500*DAY, "last_auth": now - 300*DAY,
                    "owner": "", "service_running": False},
 "sunset-agent":   {"created": now - 300*DAY, "last_auth": now - 2*DAY,
                    "owner": "", "service_running": False},
}
print(f"{'identity':18s}{'last auth (d)':>15}{'service running':>18}{'owner':>12}  finding")
print("-" * 92)
for name, i in IDENTITIES.items():
    age = (now - i["last_auth"])/DAY
    finding = ""
    if not i["service_running"] and age < 30:
        finding = "ACTIVE CREDENTIAL FOR A RETIRED SERVICE"
    elif not i["service_running"]:
        finding = "orphan — decommissioning never finished"
    elif not i["owner"]:
        finding = "no owner"
    print(f"{name:18s}{age:>15.0f}{str(i['service_running']):>18}"
          f"{i['owner'] or '—':>12}  {finding}")
print("\nRead the sunset-agent row twice. The service was retired. The identity")
print("authenticated two days ago. Somebody or something is still using it.")

## 5 · The control — two automated checks that close the loop

In [ ]:
def lifecycle_checks(identities, now, stale_days=90):
    findings = []
    for name, i in identities.items():
        age = (now - i["last_auth"])/DAY
        if not i["service_running"] and age < stale_days:
            findings.append((name, "critical",
                             "identity active for a decommissioned service"))
        elif age > stale_days:
            findings.append((name, "medium",
                             f"no authentication in {age:.0f}d — decommission it"))
        elif not i["owner"]:
            findings.append((name, "medium", "no named owner"))
    return findings

for name, sev, why in lifecycle_checks(IDENTITIES, now):
    print(f"[{sev:8s}] {name:18s} {why}")

print("\nand the manifest-diff check, for the events that change behaviour:")
SCOPE_WEIGHT = {"self": 1, "project": 3, "tenant": 8, "org": 20}
def blast(tools, gated=frozenset()):
    return sum(SCOPE_WEIGHT[s]*(1 if rev else 2) for n,s,rev in tools if n not in gated)
BEFORE = [("read_file","self",True)]
AFTER  = [("read_file","self",True), ("deploy","org",False)]
d = blast(AFTER) - blast(BEFORE)
print(f"   manifest changed: blast {blast(BEFORE)} → {blast(AFTER)} (+{d})")
print(f"   → requires re-tiering (E1.3) and a fresh SB-2 test (E1.7)")

crit = [f for f in lifecycle_checks(IDENTITIES, now) if f[1] == "critical"]
assert crit
print(f"\n{len(crit)} critical lifecycle finding(s) — each is a standing credential")
print("for something everyone believes is switched off.")

## What you just proved

Four of six lifecycle events generate no reliable record at all. The identity review flags `sunset-agent` as critical — an active credential for a decommissioned service — plus two orphans with no authentication in 300+ days. The manifest diff shows the blast radius rising from 0 to 40, requiring re-tiering and a fresh control test.

## Your turn

Query your identity provider for non-human identities whose service is retired but which authenticated in the last 30 days. Every hit is either an undocumented dependency or someone else's foothold, and you cannot tell which from the directory alone.

---

**Next → [E1.10 · The stakeholder map: who owns what](https://spbreed.github.io/cyber-commons/lessons/E1.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*